In [1]:
pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 11.6 MB/s eta 0:00:00


In [2]:
import fastf1
import wandb
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import weight_norm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from google.colab import drive
import matplotlib.pyplot as plt
import os

# Dataset Formatting

In [3]:
# logging.getLogger("fastf1").setLevel(logging.ERROR)

# # 1. Setup
# drive.mount('/content/drive')
# cache_dir = '/content/drive/MyDrive/Colab Notebooks/AI535_notebooks/Final Project/Cache/'
# if not os.path.exists(cache_dir):
#     os.makedirs(cache_dir)
# fastf1.Cache.enable_cache(cache_dir)

# # 2. Configuration
# YEARS = [2022, 2023]
# MAX_FUEL_KG = 110
# # We only keep laps where TrackStatus is '1' (Green Flag) for training pure pace?
# # OR we keep all and let the model learn. Let's keep ALL for now but add a status feature.
# REQUIRED_COLUMNS = ['RaceID', 'Driver', 'LapNumber', 'LapTime', 'TyreLife', 'Compound', 'TrackStatus', 'AirTemp', 'TrackTemp', 'Rain']

# def build_race_dataset(years):
#     all_laps = []

#     for year in years:
#         schedule = fastf1.get_event_schedule(year, include_testing=False)
#         print(f"\n=== Processing Season {year} ===")

#         for i, event in schedule.iterrows():
#             round_num = event['RoundNumber']
#             race_name = event['EventName']
#             print(f"  Round {round_num}: {race_name}...", end=" ")

#             try:
#                 # Load Race Session
#                 session = fastf1.get_session(year, round_num, 'R')
#                 session.load(laps=True, telemetry=False, weather=True, messages=False)

#                 # 1. Get Weather (Resample to match lap times if needed, or take mean)
#                 # FastF1 weather is time-series. We merge it into laps based on 'Time' (Lap End Time)
#                 weather = session.weather_data
#                 laps = session.laps

#                 if weather is None or laps is None:
#                     print("No data found.")
#                     continue

#                 # We need to map weather to laps. Simplified approach:
#                 # Use the air/track temp at the START of the lap?
#                 # Let's just use the session average for simplicity or map strictly if you want high precision.
#                 # Here we will do a 'merge_asof' to get precise temp per lap.
#                 # FastF1 uses 'Rainfall' (True/False)
#                 if 'Rainfall' in weather.columns:
#                     weather = weather.rename(columns={'Rainfall': 'Rain'})

#                 # Convert 'Time' to timedelta if it isn't already (usually it is)
#                 # We need to ensure types match for merge_asof
#                 laps['Time'] = pd.to_timedelta(laps['Time'])
#                 weather['Time'] = pd.to_timedelta(weather['Time'])

#                 # Merge Weather into Laps
#                 # We use merge_asof to find the weather condition *nearest* to the lap finish time
#                 laps = pd.merge_asof(
#                     laps.sort_values('Time'),
#                     weather[['Time', 'AirTemp', 'TrackTemp', 'Rain']],
#                     on='Time',
#                     direction='backward'
#                 )

#                 # 2. Clean Data
#                 # Convert LapTime to seconds
#                 laps['LapTime_Sec'] = laps['LapTime'].dt.total_seconds()

#                 # Drop laps with no time (formation laps, crashes)
#                 laps = laps.dropna(subset=['LapTime_Sec', 'TyreLife', 'Compound']).copy()

#                 # ---- New Features ---- START

#                 # 1. Total Laps (Denominator for Progress)
#                 # We use the max lap number recorded in the session (handles shortened races)
#                 total_laps = laps['LapNumber'].max()

#                 # 2. Race Progress (0.0 to 1.0)
#                 laps['RaceProgress'] = laps['LapNumber'] / total_laps

#                 # 3. Estimated Fuel Load (Linear burn from 110kg)
#                 # laps['FuelLoad'] = MAX_FUEL_KG * (1 - laps['RaceProgress'])

#                 # 4. Lap Time Delta (Gap to Previous Lap)
#                 # We must group by Driver so we don't calculate delta between Hamilton and Verstappen
#                 # fillna(0) handles the first lap
#                 laps['LapTimeDelta'] = laps.groupby('Driver')['LapTime_Sec'].diff().fillna(0)

#                 # 5. Sector Times
#                 laps['Sector1'] = laps['Sector1Time'].dt.total_seconds()
#                 laps['Sector2'] = laps['Sector2Time'].dt.total_seconds()
#                 laps['Sector3'] = laps['Sector3Time'].dt.total_seconds()
#                 laps = laps.dropna(subset=['Sector1Time','Sector2Time','Sector3Time']).copy()

#                 for col in ['Sector1', 'Sector2', 'Sector3']:
#                   sector_mean = laps[col].mean()
#                   sector_std = laps[col].std()
#                   laps[f"{col}_Z"] = (laps[col] - sector_mean) / sector_std

#                 # ---- New Features ---- END

#                 # 3. Per-Race Normalization (The Secret Sauce)
#                 # Calculate stats for THIS race only
#                 race_mean = laps['LapTime_Sec'].mean()
#                 race_std = laps['LapTime_Sec'].std()

#                 # Z-Score Transform
#                 laps['LapTime_Z'] = (laps['LapTime_Sec'] - race_mean) / race_std

#                 # Store metadata so we can reverse it later if needed
#                 laps['Race_Mean'] = race_mean
#                 laps['Race_Std'] = race_std
#                 laps['RaceID'] = f"{year}_{round_num}"

#                 # Select only useful columns
#                 keep_cols = ['RaceID', 'Driver', 'LapNumber', 'LapTime_Sec', 'LapTime_Z', 'LapTimeDelta', 'RaceProgress', 'Sector1', 'Sector2', 'Sector3',
#                              'Sector1_Z', 'Sector2_Z', 'Sector3_Z',
#                              'TyreLife', 'Compound', 'TrackStatus', 'TrackTemp', 'Rain', 'Team']

#                 all_laps.append(laps[keep_cols])

#                 print(f"Done. ({len(laps)} laps)")

#             except Exception as e:
#                 print(f"Skipped: {e}")
#                 continue

#     # Concatenate all races
#     full_df = pd.concat(all_laps, ignore_index=True)
#     return full_df

# # --- EXECUTION --- TRAINING DATASET
# # This will take 10-20 mins depending on cache
# df_dataset = build_race_dataset(YEARS)

# # Save to Drive immediately
# save_path = '/content/drive/MyDrive/Colab Notebooks/AI535_notebooks/Final Project/Cache/f1_race_dataset_2022_2023.csv'
# df_dataset.to_csv(save_path, index=False)
# print(f"\nDataset saved to {save_path}")
# print(f"Total Rows: {len(df_dataset)}")
# df_dataset.head()

In [4]:
# # TEST/VAL DATASET

# df_val = build_race_dataset([2024])
# df_test = build_race_dataset([2025])

# #

# # --- SPLIT STRATEGY ---
# # 2024 had 24 races.
# # Val: Rounds 1-14 (Bahrain -> Belgium)
# # Test: Rounds 15-24 (Netherlands -> Abu Dhabi)
# # VAL_ROUNDS = range(1, 15)
# # TEST_ROUNDS = range(15, 25) # Up to 24

# # # Extract RoundNumber from RaceID (format "2024_1") to split
# # df_2024['RoundNum'] = df_2024['RaceID'].apply(lambda x: int(x.split('_')[1]))

# # df_val = df_2024[df_2024['RoundNum'].isin(VAL_ROUNDS)].copy()
# # df_test = df_2024[df_2024['RoundNum'].isin(TEST_ROUNDS)].copy()

# val_path = '/content/drive/MyDrive/Colab Notebooks/AI535_notebooks/Final Project/Cache/f1_race_dataset_2024_val.csv'
# test_path = '/content/drive/MyDrive/Colab Notebooks/AI535_notebooks/Final Project/Cache/f1_race_dataset_2024_test.csv'
# df_val.to_csv(val_path, index=False)
# df_test.to_csv(test_path, index=False)

# print(f"Total Rows: {len(df_val)}")
# print(f"Total Rows: {len(df_test)}")

# Dataset Normalization

In [5]:
# # Configuration
# WINDOW_SIZE = 10  # Look at past 5 laps
# PREDICT_HORIZON = 1 # Predict next lap

# def create_sequences(df):
#     sequences = []
#     targets = []

#     # FastF1 TrackStatus is often a string ('1', '4'). convert to float.
#     df['TrackStatus'] = pd.to_numeric(df['TrackStatus'], errors='coerce').fillna(0)
#     # 1. Encoding Categorical Variables
#     # Compound: Soft, Medium, Hard, Intermediate, Wet
#     # We map them manually to ensure consistency across years
#     COMPOUND_MAP = {
#     'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4,
#     'C1': 2, 'C2': 1, 'C3': 1, 'C4': 0, 'C5': 0 # Fallbacks
#     }
#     TEAM_MAP = {
#     'Red Bull Racing': 0,
#     'Ferrari': 1,
#     'Mercedes': 2,
#     'McLaren': 3,
#     'Alpine': 4,
#     'Aston Martin': 5,
#     'Williams': 6,
#     'Haas F1 Team': 7,

#     # Rebrands
#     'AlphaTauri': 8, 'RB': 8,                 # Faenza Team
#     'Alfa Romeo': 9, 'Kick Sauber': 9, 'Sauber': 9  # Hinwil Team
# }
#     # Map Compounds
#     df['Compound_Idx'] = df['Compound'].map(COMPOUND_MAP).fillna(1) # Default to Medium if unknown

#     # Map Teams (Handle new 2024 names)
#     df['Team_Idx'] = df['Team'].map(TEAM_MAP)

#     # Error Check: If a new team name appears that isn't in our map, warn us
#     if df['Team_Idx'].isnull().any():
#         missing_teams = df[df['Team_Idx'].isnull()]['Team'].unique()
#         print(f"WARNING: Found unmapped teams: {missing_teams}. Assigning to ID 10.")
#         df['Team_Idx'] = df['Team_Idx'].fillna(10)

#     # 2. Iterate by Driver and Race
#     # We cannot slide a window across two different races!
#     grouped = df.groupby(['RaceID', 'Driver'])
#     feature_cols = ['LapTime_Z', 'LapTimeDelta', 'RaceProgress', 'Sector1_Z', 'Sector2_Z', 'Sector3_Z', 'TyreLife', 'TrackTemp', 'TrackStatus', 'Compound_Idx', 'Team_Idx']

#     for (race_id, driver), group in grouped:
#         # Sort by LapNumber to ensure time order
#         group = group.sort_values('LapNumber')

#         # We need at least window_size + 1 laps
#         if len(group) < WINDOW_SIZE + PREDICT_HORIZON:
#             continue

#         # Convert features to numpy array
#         # Features: [LapTime_Z, TyreLife, TrackTemp, TrackStatus, Compound_Idx, Team_Idx]
#         try:
#             data = group[feature_cols].astype(np.float32).values
#         except ValueError as e:
#             print(f"Error converting data for {driver} in {race_id}: {e}")
#             continue

#         # Create Windows
#         for i in range(len(group) - WINDOW_SIZE):
#             # Input: Steps i to i+k
#             seq_x = data[i : i + WINDOW_SIZE]

#             # Target: LapTime_Z at step i+k
#             target_y = data[i + WINDOW_SIZE][0] # Index 0 is LapTime_Z

#             sequences.append(seq_x)
#             targets.append(target_y)

#     return np.array(sequences), np.array(targets)

# # --- EXECUTION ---
# print("Generating Sequences...")
# X_np, y_np = create_sequences(df_dataset)

# print(f"Input Shape (X): {X_np.shape}")
# # Expected: (N_Samples, 5, 6) -> (Samples, Window_Size, Features)
# print(f"Target Shape (y): {y_np.shape}")

# # Double check the type before saving
# print(f"Array Type: {X_np.dtype}") # Should now say float32

# # 1. Generate Validation Tensors
# print("\nGenerating Validation Sequences...")
# X_val, y_val = create_sequences(df_val)

# # 2. Generate Test Tensors
# print("Generating Test Sequences...")
# X_test, y_test = create_sequences(df_test)

# # Save the Tensors
# import torch
# save_path_pt = os.path.join(cache_dir, 'f1_sequences_2022_2023.pt')
# torch.save({
#     'X': torch.tensor(X_np, dtype=torch.float32),
#     'y': torch.tensor(y_np, dtype=torch.float32)
# }, save_path_pt)
# print("Sequences saved as PyTorch tensors.")

# # 3. Save
# val_path = os.path.join(cache_dir, 'f1_val_2024.pt')
# test_path = os.path.join(cache_dir, 'f1_test_2024.pt')

# torch.save({'X': torch.tensor(X_val, dtype=torch.float32), 'y': torch.tensor(y_val, dtype=torch.float32)}, val_path)
# torch.save({'X': torch.tensor(X_test, dtype=torch.float32), 'y': torch.tensor(y_test, dtype=torch.float32)}, test_path)

# print(f"\nDONE!")
# print(f"Validation saved to {val_path} | Shape: {X_val.shape}")
# print(f"Test saved to {test_path}       | Shape: {X_test.shape}")

# TCN Layer

In [6]:
class Chomp1d(nn.Module):
    """
    Removes the padding from the 'future' side to keep the convolution causal.
    """
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        # Slices the tensor to remove the last 'chomp_size' elements
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    One block of the TCN.
    Structure: [Dilated Conv -> ReLU -> Dropout] x 2
    Plus a Residual Connection (Input + Output)
    """
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()

        # Layer 1
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding) # Remove future padding
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        # Layer 2
        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)

        # 1x1 Conv to match dimensions for Residual connection if needed
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

# TCN Network

In [7]:
class F1_TCN(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(F1_TCN, self).__init__()
        layers = []
        num_levels = len(num_channels)

        # Build the TCN Tower
        for i in range(num_levels):
            dilation_size = 2 ** i # 1, 2, 4, 8...
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]

            # Padding = (Kernel-1) * Dilation
            # This ensures the output length equals input length
            padding = (kernel_size - 1) * dilation_size

            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=padding, dropout=dropout)]

        self.network = nn.Sequential(*layers)

        # Regression Head
        # Maps the final channel dimension (e.g., 64) to 1 scalar output
        self.linear = nn.Linear(num_channels[-1], 1)

    def forward(self, x):
        # Input x shape: (Batch, Length, Features) -> e.g., (32, 8, 11)
        # TCN expects:   (Batch, Features, Length) -> e.g., (32, 11, 8)
        x = x.transpose(1, 2)

        # Pass through TCN
        y = self.network(x)

        # y shape is (Batch, Channels, Length)
        # We only care about the prediction at the LAST time step (the most recent lap)
        # This makes it "Many-to-One"
        y = y[:, :, -1]

        return self.linear(y) # Output shape: (Batch, 1)

# MLP Network

In [8]:
class F1_MLP(nn.Module):
    def __init__(self, seq_len, num_features, hidden_sizes=[128, 64, 64], dropout=0.2):
        super(F1_MLP, self).__init__()

        # MLPs require 2D input (Batch, Features), so we flatten the 3D sequence
        # Shape goes from (Batch, 8, 11) -> (Batch, 88)
        self.flatten = nn.Flatten()
        input_dim = seq_len * num_features

        layers = []
        current_dim = input_dim

        # Dynamically build the hidden layers
        for h_dim in hidden_sizes:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            current_dim = h_dim

        # Final regression head mapping to a single scalar (LapTime_Z)
        layers.append(nn.Linear(current_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # Flatten the temporal dimension
        x = self.flatten(x)
        # Pass through fully connected layers
        return self.network(x)

# Initialization of the network

In [9]:
input_channels = 11

# Hidden Layers: 3 layers expanding up to 64 channels
# Dilation will be 1, 2, 4. Receptive field = sufficient for window of 8.
channel_sizes = [32, 64, 64]

model = F1_TCN(num_inputs=input_channels, num_channels=channel_sizes, kernel_size=3, dropout=0.2)

print(model)

# Test with a dummy input to verify shapes
dummy_input = torch.randn(32, 8, 11) # Batch 32, Window 8, Features 11
output = model(dummy_input)
print(f"\nDummy Input Shape: {dummy_input.shape}")
print(f"Model Output Shape: {output.shape} (Should be [32, 1])")

F1_TCN(
  (network): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(11, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp1): Chomp1d()
      (relu1): ReLU()
      (dropout1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp2): Chomp1d()
      (relu2): ReLU()
      (dropout2): Dropout(p=0.2, inplace=False)
      (net): Sequential(
        (0): Conv1d(11, 32, kernel_size=(3,), stride=(1,), padding=(2,))
        (1): Chomp1d()
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
        (5): Chomp1d()
        (6): ReLU()
        (7): Dropout(p=0.2, inplace=False)
      )
      (downsample): Conv1d(11, 32, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TemporalBlock(
      (conv1): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (chomp1): Chomp1d()
      (relu1): Re

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


# Training and Validation

In [10]:
# --- 1. SETUP ---
# Ensure you are logged in to wandb in your terminal or notebook
# Replace key with your actual key if not using Secrets
wandb.login(key='wndb_v1_SONXY1KdahvYmQeb8gprpPTN2mU_PSj7jA4oHwpLi78iboqSE6fCcoTCI6mVQljdRTyFFXV0*****')

drive.mount('/content/drive') # Uncomment if running in a fresh session
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = '/content/drive/MyDrive/Colab Notebooks/AI535_notebooks/Final Project/Cache/'

# Define Hyperparameters Config
config = {
    'epochs': 50,
    'batch_size': 64,
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'tolerance': 0.5,
    'architecture': 'TCN',
    'dataset': 'F1_2022_2024'
}

# --- 2. HELPER FUNCTIONS ---
def load_data(path, batch_size, shuffle=True):
    data = torch.load(path)
    dataset = TensorDataset(data['X'], data['y'])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def calculate_accuracy(outputs, targets, tolerance=0.5):
    """
    Regression Accuracy: % of predictions within 'tolerance' Z-scores of truth.
    """
    diff = torch.abs(outputs - targets)
    correct = (diff < tolerance).float()
    return correct.mean().item()

def plot_regression_scatter(targets, preds, title='Prediction Correlation'):
    """
    Creates a matplotlib figure comparing True vs Predicted values.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(targets, preds, alpha=0.3, s=10)

    # Plot perfect diagonal line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()]),
    ]
    ax.plot(lims, lims, 'r-', alpha=0.75, zorder=0)
    ax.set_aspect('equal')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel('Actual Lap Time (Z-Score)')
    ax.set_ylabel('Predicted Lap Time (Z-Score)')
    ax.set_title(title)
    return fig

# --- 3. MAIN PIPELINE (TRAIN + TEST) ---

def run_experiment():
    # Initialize WandB
    with wandb.init(project="f1-laptime-tcn", config=config) as run:

        # --------------------------
        # A. DATA LOADING
        # --------------------------
        print("Loading Data...")
        train_loader = load_data(os.path.join(CACHE_DIR, 'f1_sequences_2022_2023.pt'), config['batch_size'], True)
        val_loader   = load_data(os.path.join(CACHE_DIR, 'f1_val_2024.pt'), config['batch_size'], False)
        # Load Test Data now to ensure it exists, but use it later
        test_loader  = load_data(os.path.join(CACHE_DIR, 'f1_test_2024.pt'), config['batch_size'], False)

        # --------------------------
        # B. MODEL SETUP
        # --------------------------
        if config['architecture'] == 'TCN':
            model = F1_TCN(
                num_inputs=11,
                num_channels=[64, 128, 128, 256],
                kernel_size=3,
                dropout=0.2
            ).to(device)

        elif config['architecture'] == 'MLP':
            # WINDOW_SIZE is 8, num_features is 11
            model = F1_MLP(
                seq_len=10,
                num_features=11,
                hidden_sizes=[256, 128, 128], # Scaled to roughly match TCN capacity
                dropout=0.2
            ).to(device)

        wandb.watch(model, log="all", log_freq=100)

        optimizer = optim.SGD(
            model.parameters(),
            lr=config['lr'],
            momentum=config['momentum'],
            weight_decay=config['weight_decay']
        )

        criterion = nn.HuberLoss(delta=1.0)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

        # --------------------------
        # C. TRAINING LOOP
        # --------------------------
        print("Starting Training...")
        best_val_loss = float('inf')
        global_step = 0

        for epoch in range(config['epochs']):

            # -- TRAIN --
            model.train()
            running_loss = 0.0
            running_acc = 0.0

            for i, (inputs, targets) in enumerate(train_loader):
                inputs, targets = inputs.to(device), targets.to(device).view(-1, 1)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                acc = calculate_accuracy(outputs, targets, config['tolerance'])
                running_loss += loss.item()
                running_acc += acc
                global_step += 1

                wandb.log({
                    "train/batch_loss": loss.item(),
                    "train/batch_acc": acc,
                    "train/step": global_step
                })

            avg_train_loss = running_loss / len(train_loader)
            avg_train_acc = running_acc / len(train_loader)

            # -- VALIDATE --
            model.eval()
            val_loss = 0.0
            val_acc = 0.0

            # Collectors for visualization (First batch only to save time)
            vis_preds = []
            vis_targets = []

            with torch.no_grad():
                for i, (inputs, targets) in enumerate(val_loader):
                    inputs, targets = inputs.to(device), targets.to(device).view(-1, 1)

                    outputs = model(inputs)
                    loss = criterion(outputs, targets)

                    val_loss += loss.item()
                    val_acc += calculate_accuracy(outputs, targets, config['tolerance'])

                    if i == 0: # Only grab first batch for plot
                        vis_preds = outputs.cpu().numpy()
                        vis_targets = targets.cpu().numpy()

            avg_val_loss = val_loss / len(val_loader)
            avg_val_acc = val_acc / len(val_loader)

            # -- SCHEDULER --
            current_lr = scheduler.get_last_lr()[0]
            scheduler.step()

            # -- LOGGING --
            fig = plot_regression_scatter(vis_targets.flatten(), vis_preds.flatten(), title=f'Val Epoch {epoch}')

            wandb.log({
                "train/loss": avg_train_loss,
                "train/accuracy": avg_train_acc,
                "val/loss": avg_val_loss,
                "val/accuracy": avg_val_acc,
                "train/learning_rate": current_lr,
                "val/scatter_plot": wandb.Image(fig),
                "epoch": epoch
            })
            plt.close(fig)

            print(f"Epoch {epoch+1}/{config['epochs']} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} ({avg_val_acc:.1%})")

            # -- CHECKPOINTING --
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                checkpoint_name = f"best_model_{config['architecture']}.pth"
                torch.save(model.state_dict(), os.path.join(CACHE_DIR, checkpoint_name))

        # --------------------------
        # D. TEST EVALUATION
        # --------------------------
        print("\nTraining Complete. Starting Test Evaluation (Late 2024)...")

        # Load Best Model Weights
        checkpoint_name = f"best_model_{config['architecture']}.pth"
        best_model_path = os.path.join(CACHE_DIR, checkpoint_name)

        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.eval()

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device).view(-1, 1)
                outputs = model(inputs)

                all_preds.append(outputs.cpu().numpy())
                all_targets.append(targets.cpu().numpy())

        # Concatenate
        y_pred = np.concatenate(all_preds).flatten()
        y_true = np.concatenate(all_targets).flatten()

        # Metrics
        test_mse = np.mean((y_pred - y_true) ** 2)
        test_mae = np.mean(np.abs(y_pred - y_true))

        # Accuracy Calculation (Manual for full numpy array)
        diff = np.abs(y_pred - y_true)
        test_acc = np.mean(diff < config['tolerance'])

        print("="*30)
        print(f"FINAL TEST RESULTS (Best Model)")
        print(f"MSE: {test_mse:.4f}")
        print(f"MAE: {test_mae:.4f}")
        print(f"Accuracy: {test_acc:.2%}")
        print("="*30)

        # Final Plots
        fig_scatter = plot_regression_scatter(y_true, y_pred, title='Final Test: Predicted vs Actual')

        # Residual Plot
        fig_resid, ax_resid = plt.subplots(figsize=(10, 4))
        residuals = y_pred - y_true
        ax_resid.hist(residuals, bins=50, color='purple', alpha=0.7)
        ax_resid.axvline(0, color='k', linestyle='--')
        ax_resid.set_title('Final Test: Residual Error Distribution')
        ax_resid.set_xlabel('Error (Z-Score)')

        # Log Final Test Metrics to WandB Summary
        wandb.log({
            "test/mse": test_mse,
            "test/mae": test_mae,
            "test/accuracy": test_acc,
            "test/scatter_plot": wandb.Image(fig_scatter),
            "test/residual_plot": wandb.Image(fig_resid)
        })

        plt.close(fig_scatter)
        plt.close(fig_resid)
        print("Test results logged to WandB.")

# --- 4. EXECUTION ---
run_experiment()

config['architecture']  = 'MLP'
print("="*30)
print(f'Switching to MLP')
run_experiment()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: modim (modim-oregon-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Mounted at /content/drive


Loading Data...


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Starting Training...
Epoch 1/50 | Train: 0.2374 | Val: 0.2004 (85.2%)
Epoch 2/50 | Train: 0.1902 | Val: 0.2281 (69.6%)
Epoch 3/50 | Train: 0.1509 | Val: 0.1301 (87.0%)
Epoch 4/50 | Train: 0.1341 | Val: 0.1288 (87.5%)
Epoch 5/50 | Train: 0.1251 | Val: 0.1129 (88.4%)
Epoch 6/50 | Train: 0.1190 | Val: 0.1143 (88.9%)
Epoch 7/50 | Train: 0.1145 | Val: 0.1133 (89.2%)
Epoch 8/50 | Train: 0.1143 | Val: 0.1106 (89.1%)
Epoch 9/50 | Train: 0.1093 | Val: 0.1085 (89.2%)
Epoch 10/50 | Train: 0.1084 | Val: 0.1097 (89.6%)
Epoch 11/50 | Train: 0.1072 | Val: 0.1133 (89.3%)
Epoch 12/50 | Train: 0.1055 | Val: 0.1011 (90.4%)
Epoch 13/50 | Train: 0.1036 | Val: 0.1129 (88.6%)
Epoch 14/50 | Train: 0.1021 | Val: 0.1108 (88.7%)
Epoch 15/50 | Train: 0.1019 | Val: 0.0988 (90.5%)
Epoch 16/50 | Train: 0.1008 | Val: 0.0993 (90.5%)
Epoch 17/50 | Train: 0.1004 | Val: 0.1044 (90.0%)
Epoch 18/50 | Train: 0.0995 | Val: 0.1070 (89.8%)
Epoch 19/50 | Train: 0.0970 | Val: 0.1014 (90.0%)
Epoch 20/50 | Train: 0.0972 | Val: 0.1

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
test/accuracy,▁
test/mae,▁
test/mse,▁
train/accuracy,▁▁▂▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████
train/batch_acc,▁▄▃▆▅▅▅▆▆▆▄▆▆▅▅▅▆█▇▆▅▄▅▃▄▅▃▄▅▅▇▆▃▆▅▆▅▇▄▇
train/batch_loss,█▅▅▆▃▄▅▃▃▂▄▃▃▃▂▃▃▄▃▄▃▃▃▃▄▃▂▃▄▁▂▁▃▂▂▃▂▂▂▂
train/learning_rate,███████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/loss,█▆▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step,▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
+2,...


Switching to MLP


Loading Data...
Starting Training...
Epoch 1/50 | Train: 0.2368 | Val: 0.1970 (85.0%)
Epoch 2/50 | Train: 0.2029 | Val: 0.1802 (87.2%)
Epoch 3/50 | Train: 0.1874 | Val: 0.1707 (87.2%)
Epoch 4/50 | Train: 0.1794 | Val: 0.1666 (87.9%)
Epoch 5/50 | Train: 0.1746 | Val: 0.1668 (88.3%)
Epoch 6/50 | Train: 0.1719 | Val: 0.1540 (89.2%)
Epoch 7/50 | Train: 0.1674 | Val: 0.1549 (88.5%)
Epoch 8/50 | Train: 0.1670 | Val: 0.1513 (89.1%)
Epoch 9/50 | Train: 0.1646 | Val: 0.1663 (88.0%)
Epoch 10/50 | Train: 0.1611 | Val: 0.1530 (88.5%)
Epoch 11/50 | Train: 0.1586 | Val: 0.1416 (89.1%)
Epoch 12/50 | Train: 0.1561 | Val: 0.1541 (88.5%)
Epoch 13/50 | Train: 0.1537 | Val: 0.1345 (89.0%)
Epoch 14/50 | Train: 0.1543 | Val: 0.1357 (89.2%)
Epoch 15/50 | Train: 0.1484 | Val: 0.1583 (87.8%)
Epoch 16/50 | Train: 0.1496 | Val: 0.1300 (88.8%)
Epoch 17/50 | Train: 0.1476 | Val: 0.1407 (88.4%)
Epoch 18/50 | Train: 0.1455 | Val: 0.1417 (89.0%)
Epoch 19/50 | Train: 0.1449 | Val: 0.1323 (89.2%)
Epoch 20/50 | Train: 0

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
test/accuracy,▁
test/mae,▁
test/mse,▁
train/accuracy,▁▂▃▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train/batch_acc,▁▃▅▃▃▃▄▂▅▂▃▅▅▅▅▅▃▁▅▃▄▃▅▃▅▅▄▆▅▆▅▂▆▆▃▅▇▆▅█
train/batch_loss,█▆█▄▄▃▃▃▁▅▅▃▃▃▃▆▂▇▃▁▂▂▃▃▃▃▂▂▁▃▁▃▁▄▁▂▃▂▄▃
train/learning_rate,███████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train/step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▆▆▆▆▆▆▆▇▇▇▇▇▇██
+2,...


In [11]:
from google.colab import runtime
runtime.unassign()